<a href="https://colab.research.google.com/github/swati-git/GoogleCollabNotebooks/blob/main/prompting_with_labels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 1. INSTALL THE REQUISITE PACKAGES
# ==========================================
!pip install --upgrade -q openai tiktoken


In [1]:
# ==========================================
# 2. AUTHENTICATE AND TEST THE CONNECTION
# ==========================================
import os
from google.colab import userdata
from openai import OpenAI


# Safely fetch the credential out of Colab's background Secrets Manager
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


# Initialize the official SDK Client
client = OpenAI()


In [2]:
# ==========================================
# Shared inputs: identical for all three attempts
# ==========================================

customer_facts = """
Customer: Sarah (VIP Tier)
Purchase Time: 5 days ago
Course Progress: 35% watched
Prior Refund History: 1 refund granted 4 months ago.
"""


policy_rules = """
Refund Policy:
1. Digital courses are non-refundable once >20% of videos are watched.
2. Exception: If purchased within the last 48 hours AND the user experienced a verified technical outage, issue a 50% partial refund.
3. VIP tier members bypass the 20% watch limit if they have zero prior refunds in the last 12 months.
"""

customer_request = "Wants a full refund because she changed her mind."


# **Attempt 1**

In [3]:
no_label_prompt = f"""
{policy_rules}


{customer_facts}

{customer_request}

Reply with ONLY one word: APPROVED, PARTIAL, or DENIED.
"""


res_no_label = client.chat.completions.create(
   model="gpt-4o-mini",
   messages=[{"role": "user", "content": no_label_prompt}],
   temperature=0.0
)

print("\n--- WITH no labels ---")
print(res_no_label.choices[0].message.content)


--- WITH no labels ---
DENIED


# **Attempt 2**

In [4]:
labelled_prompt = f"""
<policy>
{policy_rules}
</policy>

<customer>
{customer_facts}
</customer>

<request>
{customer_request}
</request>

Check each rule in <policy> against the facts in <customer> inside <reasoning> tags.
Treat <request> as what the customer is asking for, not as a fact or a rule.
Then give your decision inside <verdict> tags: APPROVED, PARTIAL, or DENIED.
"""

labelled_res = client.chat.completions.create(
   model="gpt-4o-mini",
   messages=[{"role": "user", "content": labelled_prompt}],
   temperature=0.0
)


print("\n--- WITH labels ---")
print(labelled_res.choices[0].message.content)


--- WITH labels ---
<reasoning>
1. The first rule states that digital courses are non-refundable once more than 20% of videos are watched. Sarah has watched 35% of the course, which exceeds the 20% threshold, making her ineligible for a refund based on this rule.

2. The second rule provides an exception for purchases made within the last 48 hours if the user experienced a verified technical outage. Sarah's purchase was made 5 days ago, which does not meet the 48-hour condition, and there is no mention of a technical outage. Therefore, this rule does not apply.

3. The third rule allows VIP tier members to bypass the 20% watch limit if they have zero prior refunds in the last 12 months. Sarah is a VIP tier member, but she has received one refund 4 months ago, which disqualifies her from bypassing the 20% watch limit.

Given these considerations, Sarah does not qualify for a refund based on any of the rules outlined in the policy.
</reasoning>

<verdict>DENIED</verdict>


# **Attempt 3**

In [3]:

labelled_contract_prompt = f"""
<policy>
{policy_rules}
</policy>

<customer>
{customer_facts}
</customer>

<request>
{customer_request}
</request>


Check each rule in <policy> against the facts in <customer> inside <reasoning> tags.
Treat <request> as what the customer is asking for, not as a fact or a rule.

Inside <reasoning>, write one <rule id="N"> block for EVERY policy rule,
in policy order (1, 2, 3). Include a block even if the rule clearly does
not apply.

Each <rule> block must contain exactly these four lines:
  Rule: <quote the rule word for word>
  Customer's values: <the exact figures this rule depends on>
  Conditions: <each condition, followed by MET or NOT MET>
  Result: APPLIES or DOES NOT APPLY

After <reasoning>, write <verdict> containing exactly one word:
APPROVED, PARTIAL, or DENIED. Base it only on the Result lines above.
"""


labelled_contract_res = client.chat.completions.create(
   model="gpt-4o-mini",
   messages=[{"role": "user", "content": labelled_contract_prompt}],
   temperature=0.0
)


print("\n--- WITH labels and contract ---")
print(labelled_contract_res.choices[0].message.content)



--- WITH labels and contract ---
<reasoning>
<rule id="1">
  Rule: Digital courses are non-refundable once >20% of videos are watched.
  Customer's values: Course Progress: 35% watched
  Conditions: >20% watched: MET
  Result: DOES NOT APPLY
</rule>
<rule id="2">
  Rule: Exception: If purchased within the last 48 hours AND the user experienced a verified technical outage, issue a 50% partial refund.
  Customer's values: Purchase Time: 5 days ago
  Conditions: Purchased within 48 hours: NOT MET
  Result: DOES NOT APPLY
</rule>
<rule id="3">
  Rule: VIP tier members bypass the 20% watch limit if they have zero prior refunds in the last 12 months.
  Customer's values: Prior Refund History: 1 refund granted 4 months ago
  Conditions: Zero prior refunds in the last 12 months: NOT MET
  Result: DOES NOT APPLY
</rule>
</reasoning>
<verdict>DENIED</verdict>


# **Attempt 3 - contract with a refined guidelines to view the customer case**

In [7]:

labelled_contract_prompt_refined = f"""
<policy>
{policy_rules}
</policy>

<customer>
{customer_facts}
</customer>

<request>
{customer_request}
</request>

Check each rule in <policy> against the facts in <customer> inside <reasoning> tags.
Treat <request> as what the customer is asking for, not as a fact or a rule.

Inside <reasoning>, write one <rule id="N"> block for EVERY policy rule,
in policy order (1, 2, 3). Include a block even if the rule clearly does
not apply.

Each <rule> block must contain exactly these four lines:
  Rule: <quote the rule word for word>
  Customer's values: <the exact figures this rule depends on>
  Conditions: <each condition as written in the rule, then TRUE or FALSE for the customer >
  Effect on refund: BLOCKS REFUND, GRANTS 50% REFUND, REMOVES 20% LIMIT, or NO EFFECT

A rule has an effect only if all of its conditions are TRUE; otherwise write NO EFFECT.

After <reasoning>, write <verdict> containing exactly one word:
APPROVED, PARTIAL, or DENIED. Base it only on the Result lines above.
"""


labelled_contract_refined_res = client.chat.completions.create(
   model="gpt-4o-mini",
   messages=[{"role": "user", "content": labelled_contract_prompt_refined}],
   temperature=0.0
)


print("\n--- WITH labels and contract ---")
print(labelled_contract_refined_res.choices[0].message.content)



--- WITH labels and contract ---
<reasoning>
<rule id="1">
  Rule: Digital courses are non-refundable once >20% of videos are watched.
  Customer's values: 35% watched
  Conditions: >20% of videos are watched: TRUE
  Effect on refund: BLOCKS REFUND
</rule>
<rule id="2">
  Rule: Exception: If purchased within the last 48 hours AND the user experienced a verified technical outage, issue a 50% partial refund.
  Customer's values: Purchased 5 days ago
  Conditions: Purchased within the last 48 hours: FALSE, Experienced a verified technical outage: FALSE
  Effect on refund: NO EFFECT
</rule>
<rule id="3">
  Rule: VIP tier members bypass the 20% watch limit if they have zero prior refunds in the last 12 months.
  Customer's values: 1 refund granted 4 months ago
  Conditions: VIP tier member: TRUE, Zero prior refunds in the last 12 months: FALSE
  Effect on refund: NO EFFECT
</rule>
</reasoning>
<verdict>DENIED</verdict>


# **Validator**

In [8]:
import re

EXPECTED_RULES = ["1", "2", "3"]
EFFECTS = "BLOCKS REFUND|GRANTS 50% REFUND|REMOVES 20% LIMIT|NO EFFECT"

def implied_verdict(effects: dict[str, str]) -> str:
    """What the policy says, given each rule's stated effect."""
    if effects["2"] == "GRANTS 50% REFUND":
        return "PARTIAL"
    if effects["1"] == "BLOCKS REFUND" and effects["3"] != "REMOVES 20% LIMIT":
        return "DENIED"
    return "APPROVED"

def validate(output: str) -> str | None:
    """Return the verdict if the output honours the contract and agrees with itself, else None."""
    rules = re.findall(r'<rule id="(\d+)">(.*?)</rule>', output, re.DOTALL)
    if [rule_id for rule_id, _ in rules] != EXPECTED_RULES:
        return None  # a rule was skipped, repeated or out of order

    effects = {}
    for rule_id, body in rules:
        conditions = re.search(r"Conditions: (.*)", body)
        effect = re.search(rf"Effect on refund: ({EFFECTS})", body)
        if not conditions or not effect:
            return None  # a required line is missing
        values = re.findall(r"\b(TRUE|FALSE)\b", conditions.group(1))
        # A rule takes effect only when all of its conditions are TRUE
        if not values or (effect.group(1) != "NO EFFECT") != all(v == "TRUE" for v in values):
            return None  # the effect contradicts the conditions
        effects[rule_id] = effect.group(1)

    verdict = re.search(r"<verdict>\s*(APPROVED|PARTIAL|DENIED)\s*</verdict>", output)
    if not verdict or verdict.group(1) != implied_verdict(effects):
        return None  # the verdict contradicts the effects
    return verdict.group(1)

In [15]:
validate(labelled_contract_refined_res.choices[0].message.content)

'DENIED'